# Project 4 — Diabetes Risk Assessment

**SciEncephalon AI · Summer Intern Series 2026**

**Author:** Reagan Lundy

## Goal
Estimate **Type-2 diabetes risk** from non-invasive inputs (age, BMI, family history, blood pressure, …) and generate an LLM-style personalized "next steps" message.

> **Not medical advice.** Education only. Real screening uses A1C / fasting glucose. We are *not* clinically diagnosing.

## Why this project
- Real-world tabular ML on a famous dataset (Pima Indians).
- You'll handle **missing data**, **class imbalance**, and **calibration** — three core issues in clinical ML.
- You'll combine ML with an LLM-stub to write personalized advice, a pattern that's everywhere in modern healthcare apps.

## Tech Stack
| Layer | Tool |
|---|---|
| Data | Pima Indians Diabetes Dataset (UCI / Kaggle) |
| Modeling | Gradient-Boosted Trees (sklearn `HistGradientBoosting` — no XGBoost install needed) |
| Calibration | `CalibratedClassifierCV` |
| Visualization | **PyEcharts** — gauge, calibration curve, bar |


## 1. Setup

In [1]:
# !pip install pandas scikit-learn pyecharts numpy --quiet
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Gauge, Pie, Page
np.random.seed(42)
print("ready ✓")

ready ✓


## 2. Load Data

We try the public OpenML mirror; if unreachable, synthesize. Schema mirrors Pima exactly so downstream code is unchanged.

In [2]:
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
COLS = ["pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","dpf","age","outcome"]
def load():
    try:
        df = pd.read_csv(URL, names=COLS)
        return df
    except Exception as e:
        print("Falling back to synthetic data:", e)
        n=768; rng=np.random.default_rng(0)
        df = pd.DataFrame({
            "pregnancies": rng.integers(0, 12, n),
            "glucose": rng.normal(120, 32, n).clip(50, 220).round().astype(int),
            "blood_pressure": rng.normal(72, 14, n).clip(40, 130).round().astype(int),
            "skin_thickness": rng.normal(22, 14, n).clip(0, 80).round().astype(int),
            "insulin": rng.gamma(2, 60, n).round().astype(int),
            "bmi": rng.normal(31, 7, n).clip(15, 60).round(1),
            "dpf": rng.gamma(2, 0.3, n).round(3),
            "age": rng.integers(20, 75, n),
        })
        z = 0.03*(df.glucose-120) + 0.05*(df.bmi-30) + 0.02*(df.age-40) + 0.5*df.dpf
        df["outcome"] = (rng.random(n) < 1/(1+np.exp(-z))).astype(int)
        return df
df = load()
# Pima encodes "missing" as 0 in several physically-impossible-zero columns
for c in ["glucose","blood_pressure","skin_thickness","insulin","bmi"]:
    df[c] = df[c].replace(0, np.nan)
df.fillna(df.median(numeric_only=True), inplace=True)
print(df.shape, "  positives:", df.outcome.mean().round(3)); df.head()

(768, 9)   positives: 0.349


,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,dpf,age,outcome
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1


## 3. Class Imbalance — Visualize It First

In [3]:
counts = df["outcome"].value_counts().to_dict()
pie = (
    Pie(init_opts=opts.InitOpts(width="500px", height="400px"))
    .add("", [("No diabetes", counts.get(0,0)), ("Diabetes", counts.get(1,0))],
         radius=["35%","65%"])
    .set_global_opts(title_opts=opts.TitleOpts(title="Class Balance"))
    .set_series_opts(label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)"))
)
pie.render_notebook()

## 4. Train

In [4]:
X = df.drop(columns=["outcome"]); y = df["outcome"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
base = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_depth=5)
clf = CalibratedClassifierCV(base, method="isotonic", cv=5)
clf.fit(Xtr, ytr)

probs = clf.predict_proba(Xte)[:,1]
preds = (probs >= 0.5).astype(int)
print(f"AUC = {roc_auc_score(yte, probs):.3f}")
print(classification_report(yte, preds))

AUC = 0.815
              precision    recall  f1-score   support

           0       0.78      0.84      0.81       125
           1       0.66      0.57      0.61        67

    accuracy                           0.74       192
   macro avg       0.72      0.70      0.71       192
weighted avg       0.74      0.74      0.74       192



## 5. Calibration Curve — Are 60% Predictions Actually 60%?

In [5]:
prob_true, prob_pred = calibration_curve(yte, probs, n_bins=10, strategy="quantile")
xs = [round(float(p), 3) for p in prob_pred]
ys = [round(float(p), 3) for p in prob_true]

line = (
    Line(init_opts=opts.InitOpts(width="700px", height="460px"))
    .add_xaxis(xs)
    .add_yaxis("Model", ys, is_smooth=True,
               linestyle_opts=opts.LineStyleOpts(width=3, color="#3b82f6"),
               symbol_size=10)
    .add_yaxis("Perfect", xs,
               linestyle_opts=opts.LineStyleOpts(type_="dashed", color="#9ca3af"))
    .set_global_opts(
        title_opts=opts.TitleOpts(title="Calibration Curve",
                                   subtitle="Closer to dashed line = better"),
        xaxis_opts=opts.AxisOpts(type_="value", name="Predicted probability", min_=0, max_=1),
        yaxis_opts=opts.AxisOpts(type_="value", name="Observed frequency", min_=0, max_=1),
    )
)
line.render_notebook()

## 6. Threshold Tuning — Sensitivity vs Specificity

In [6]:
thresholds = np.linspace(0.1, 0.9, 17)
sens, spec = [], []
for t in thresholds:
    p = (probs >= t).astype(int)
    cm = confusion_matrix(yte, p)
    tn, fp, fn, tp = cm.ravel()
    sens.append(round(tp/(tp+fn) if (tp+fn) else 0, 3))
    spec.append(round(tn/(tn+fp) if (tn+fp) else 0, 3))

xs = [round(t,2) for t in thresholds]
line = (
    Line(init_opts=opts.InitOpts(width="800px", height="440px"))
    .add_xaxis(xs)
    .add_yaxis("Sensitivity", sens, is_smooth=True,
               linestyle_opts=opts.LineStyleOpts(width=3, color="#ef4444"))
    .add_yaxis("Specificity", spec, is_smooth=True,
               linestyle_opts=opts.LineStyleOpts(width=3, color="#10b981"))
    .set_global_opts(
        title_opts=opts.TitleOpts(title="Sensitivity / Specificity vs Threshold",
                                   subtitle="Pick the trade-off appropriate for the use case"),
        xaxis_opts=opts.AxisOpts(name="threshold"),
        yaxis_opts=opts.AxisOpts(name="rate", min_=0, max_=1),
    )
)
line.render_notebook()

## 7. Personalized Lifestyle Coach (LLM Stub)

We turn the prediction + the most-actionable inputs into an empathetic, plain-English message. Replace the stub with an OpenAI/Claude call when ready.

In [7]:
ACTIONABLE = {
    "bmi": ("Body Mass Index", "consider a sustainable plan to reach BMI 18.5-24.9"),
    "glucose": ("Plasma glucose", "ask your provider about an A1C test for clarity"),
    "blood_pressure": ("Diastolic BP", "track BP weekly; reducing sodium can help"),
}

def coach(patient: pd.Series, risk: float):
    flags = []
    if patient["bmi"] > 30: flags.append(ACTIONABLE["bmi"])
    if patient["glucose"] > 140: flags.append(ACTIONABLE["glucose"])
    if patient["blood_pressure"] > 85: flags.append(ACTIONABLE["blood_pressure"])
    bullet = "\n".join(f"• **{n}**: {a}" for n, a in flags) if flags else "• Inputs look in healthy ranges — keep it up!"
    band = "lower" if risk<0.2 else "moderate" if risk<0.5 else "higher"
    return (
        f"Estimated risk band: **{band}** ({risk:.0%}).\n\n"
        f"Areas to focus on:\n{bullet}\n\n"
        f"_This is an educational estimate, not a diagnosis. "
        f"Talk to a clinician for proper screening (A1C / fasting glucose)._"
    )

idx = 7
patient = Xte.iloc[idx]
risk = float(clf.predict_proba(patient.values.reshape(1,-1))[0, 1])
print(coach(patient, risk))

Estimated risk band: **lower** (0%).

Areas to focus on:
• Inputs look in healthy ranges — keep it up!

_This is an educational estimate, not a diagnosis. Talk to a clinician for proper screening (A1C / fasting glucose)._


/Users/ndingari/Dropbox/SciEncephalon/Internship SJ/2026/intern_projects/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/Users/ndingari/Dropbox/SciEncephalon/Internship SJ/2026/intern_projects/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/Users/ndingari/Dropbox/SciEncephalon/Internship SJ/2026/intern_projects/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/Users/ndingari/Dropbox/SciEncephalon/Internship SJ/2026/intern_projects/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid f

In [8]:
# Risk gauge
gauge = (
    Gauge(init_opts=opts.InitOpts(width="500px", height="400px"))
    .add("Risk", [("Diabetes risk", round(risk*100,1))],
         axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(
             color=[(0.3, "#10b981"), (0.6, "#f59e0b"), (1, "#ef4444")], width=24)))
    .set_global_opts(title_opts=opts.TitleOpts(title="Estimated 5-yr Diabetes Risk"))
)
gauge.render_notebook()

## 8. Wrap-up

### What you built
- Calibrated gradient-boosted classifier with proper missing-data handling.
- Calibration curve and threshold-vs-sens/spec analysis — pro-level evaluation.
- An LLM-stub coach that produces personalized, disclaimer-bounded advice.

### Stretch goals
1. **Switch to BRFSS public dataset (CDC)** for 400k+ rows.
2. **Add SHAP** force plots per patient.
3. **Streamlit deployment** with sliders → live risk + coach message.
4. **Fairness audit** by ethnicity (where labels available).
5. **Diet plan generator** via the actual OpenAI/Claude API once approved.
